In [15]:
import re
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DATA_PATH = Path('bengaluru_house_prices.csv')
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)


In [16]:
df = pd.read_csv(DATA_PATH).copy()


def parse_total_sqft(value):
    text = str(value).strip()
    if pd.isna(value):
        return np.nan
    if '-' in text:
        parts = [part.strip() for part in text.split('-')]
        if len(parts) == 2:
            try:
                return (float(parts[0]) + float(parts[1])) / 2.0
            except ValueError:
                return np.nan
    match = re.search(r'([0-9]+(?:\.[0-9]+)?)', text)
    return float(match.group(1)) if match else np.nan


def parse_bhk(value):
    match = re.search(r'([0-9]+)', str(value))
    return float(match.group(1)) if match else np.nan


df['bhk'] = df['size'].apply(parse_bhk)
df['total_sqft'] = df['total_sqft'].apply(parse_total_sqft)
df['bath'] = pd.to_numeric(df['bath'], errors='coerce')
df['balcony'] = pd.to_numeric(df['balcony'], errors='coerce')
df['price'] = pd.to_numeric(df['price'], errors='coerce')
df['location'] = df['location'].fillna('Unknown').astype(str).str.strip()
df['area_type'] = df['area_type'].fillna('Unknown').astype(str).str.strip()
df['availability'] = df['availability'].fillna('Unknown').astype(str).str.strip()

df = df.dropna(subset=['price', 'total_sqft', 'bhk', 'bath'])
df = df[(df['price'] > 0) & (df['total_sqft'] > 0)]
df['sqft_per_bhk'] = df['total_sqft'] / df['bhk']
df['bath_per_bhk'] = df['bath'] / df['bhk']
df = df[df['sqft_per_bhk'].replace([np.inf, -np.inf], np.nan).notna()]

X = df[['area_type', 'availability', 'location', 'balcony', 'bath', 'bhk', 'total_sqft', 'sqft_per_bhk', 'bath_per_bhk']].copy()
y = df['price'].astype(float)

categorical_features = ['area_type', 'availability', 'location']
numeric_features = ['balcony', 'bath', 'bhk', 'total_sqft', 'sqft_per_bhk', 'bath_per_bhk']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
preprocessor = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]), numeric_features),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
    ]), categorical_features),
])

X_train_prepared = preprocessor.fit_transform(X_train)
X_test_prepared = preprocessor.transform(X_test)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train.to_frame()).ravel()

model = keras.Sequential([
    layers.Input(shape=(X_train_prepared.shape[1],)),
    layers.Dense(256, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.30),
    layers.Dense(128, activation='relu', kernel_regularizer=keras.regularizers.l2(1e-4)),
    layers.BatchNormalization(),
    layers.Dropout(0.20),
    layers.Dense(64, activation='relu'),
    layers.Dense(1),
])

model.compile(optimizer=keras.optimizers.Adam(learning_rate=5e-4), loss='mse', metrics=['mae'])
history = model.fit(
    X_train_prepared,
    y_train_scaled,
    validation_split=0.2,
    epochs=200,
    batch_size=64,
    verbose=0,
    callbacks=[
        keras.callbacks.EarlyStopping(patience=20, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(patience=8, factor=0.5, min_lr=1e-5),
    ],
)

predictions = y_scaler.inverse_transform(model.predict(X_test_prepared, verbose=0)).ravel()
print({'rmse': np.sqrt(mean_squared_error(y_test, predictions)), 'mae': mean_absolute_error(y_test, predictions), 'r2': r2_score(y_test, predictions)})

baseline = RandomForestRegressor(n_estimators=500, min_samples_leaf=2, random_state=RANDOM_STATE, n_jobs=-1)
baseline.fit(X_train_prepared, y_train)
feature_names = preprocessor.get_feature_names_out()
importance_rows = []
for feature_name, importance in zip(feature_names, baseline.feature_importances_):
    if feature_name.startswith('num__'):
        importance_rows.append((feature_name.replace('num__', ''), importance))
    else:
        encoded_name = feature_name.replace('cat__', '')
        matched_group = next(feature for feature in categorical_features if encoded_name.startswith(feature + '_'))
        importance_rows.append((matched_group, importance))
importance_df = pd.DataFrame(importance_rows, columns=['feature', 'importance']).groupby('feature', as_index=False)['importance'].sum().sort_values('importance', ascending=False)
top_feature = importance_df.iloc[0]['feature']
print('Top feature:', top_feature)
importance_df.head(10)


{'rmse': np.float64(108.34806038038188), 'mae': np.float64(33.81515048489481), 'r2': 0.43962772475347844}
Top feature: total_sqft


,feature,importance
8,total_sqft,0.643026
7,sqft_per_bhk,0.091505
3,bath,0.069759
6,location,0.061270
2,balcony,0.036931
4,bath_per_bhk,0.031864
0,area_type,0.027443
5,bhk,0.019206
1,availability,0.018995
